In [ ]:
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, roc_curve, accuracy_score)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
## my input features
alignment_matrix = np.load("./4_2e-05_model_baseline/alignment_matrix_M.npy")

In [ ]:
sns.heatmap(alignment_matrix)
plt.show()

In [ ]:
with open('./4_2e-05_model_baseline/predictions.json', 'r') as file:
    data = json.load(file)

In [ ]:
ground_truth = np.array(data["indicator_vector_true"])

In [ ]:
# For each pseudo-expert 
# The distribution of the alignment scores (for true point and non true points)
# The average +-std alignment score  (for true points and non true points)
# No of projections >0  and projections <0 (for true and non true points)
# No of projections <0 and projections<0 (for  true and non true points)

In [ ]:
# Overall picture of all datapoints 

# No of points and their corresponding projections distribution (projections >0 is capped by the no of pseudo experts)
#(for true point and non true point)- make it general function for projection>epsilon (I guess)

# No of points and their corresponding projections distribution (projections <0 is capped by the no of pseudo experts)
# (for true point and non true point) -  make it general function for projection>epsilon (I guess)

# No of points and their Average alignment score (for true point and non true point)

In [ ]:
def plot_average_alignment(alignment_matrix,ground_truth):

    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
        
    # Average alignment scores
    member_avg = np.mean(alignment_matrix[member_mask], axis=1)
    non_member_avg = np.mean(alignment_matrix[non_member_mask], axis=1)
    
    axes[0].hist(member_avg, bins=30, alpha=0.7, label='Members', density=True)
    axes[0].hist(non_member_avg, bins=30, alpha=0.7, label='Non-members', density=True)
    axes[0].set_title('Distribution of Average Alignment Scores')
    axes[0].set_xlabel('Average Alignment Score')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot comparison
    data_to_plot = [member_avg, non_member_avg]
    axes[1].boxplot(data_to_plot, labels=['Members', 'Non-members'])
    axes[1].set_title('Average Alignment Score Comparison')
    axes[1].set_ylabel('Average Alignment Score')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
alignment_matrix.shape

In [ ]:
def plot_projection_distributions(alignment_matrix, ground_truth, epsilon=0.0):
    """
    Plot distribution of positive/negative projections per sample
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: Binary array
        epsilon: Threshold for positive/negative (default 0)
    """
    
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    # Count projections > epsilon and < -epsilon for each sample
    pos_counts_members = np.sum(alignment_matrix[member_mask] > epsilon, axis=1)
    neg_counts_members = np.sum(alignment_matrix[member_mask] < epsilon, axis=1)
    
    pos_counts_non_members = np.sum(alignment_matrix[non_member_mask] > epsilon, axis=1)
    neg_counts_non_members = np.sum(alignment_matrix[non_member_mask] < epsilon, axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 10))
    
    # Positive projections
    axes[0].hist(pos_counts_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Members', density=True,align='mid')
    axes[0].hist(pos_counts_non_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Non-members', density=True,align='mid')
    axes[0].set_title(f'Distribution of Positive Projections (>{epsilon})')
    axes[0].set_xlabel('Number of Positive Projections')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    # Set x-axis ticks explicitly
    axes[1].set_xticks(range(alignment_matrix.shape[1]+1))
    axes[1].set_xticklabels(range(alignment_matrix.shape[1]+1))
    
    # Negative projections
    axes[1].hist(neg_counts_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Members', density=True,align='mid')
    axes[1].hist(neg_counts_non_members, bins=range(alignment_matrix.shape[1]+2), 
                   alpha=0.7, label='Non-members', density=True,align='mid')
    axes[1].set_title(f'Distribution of Negative Projections (<{epsilon})')
    axes[1].set_xlabel('Number of Negative Projections')
    axes[1].set_ylabel('Density')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
        # Set x-axis ticks explicitly
    axes[1].set_xticks(range(alignment_matrix.shape[1]+1))
    axes[1].set_xticklabels(range(alignment_matrix.shape[1]+1))
    
    
    # Print summary statistics
    print("\n" + "="*60)
    print(f"PROJECTION DISTRIBUTION SUMMARY (threshold = {epsilon})")
    print("="*60)
    
    print(f"Positive projections (>{epsilon}):")
    print(f"  Members     - Mean: {np.mean(pos_counts_members):5.2f} ± {np.std(pos_counts_members):4.2f}")
    print(f"  Non-members - Mean: {np.mean(pos_counts_non_members):5.2f} ± {np.std(pos_counts_non_members):4.2f}")
    
    print(f"\nNegative projections (<{epsilon}):")
    print(f"  Members     - Mean: {np.mean(neg_counts_members):5.2f} ± {np.std(neg_counts_members):4.2f}")
    print(f"  Non-members - Mean: {np.mean(neg_counts_non_members):5.2f} ± {np.std(neg_counts_non_members):4.2f}")


In [ ]:
def analyze_alignment_matrix(alignment_matrix, ground_truth):
    """
    Comprehensive analysis of alignment matrix
    
    Args:
        alignment_matrix: Shape (n_samples, n_pseudo_experts)
        ground_truth: Binary array (1=member, 0=non-member)
    """
    
    # Separate data by membership
    member_mask = ground_truth == 1
    non_member_mask = ground_truth == 0
    
    member_scores = alignment_matrix[member_mask]
    non_member_scores = alignment_matrix[non_member_mask]
    
    n_pseudo_experts = alignment_matrix.shape[1]
    
    print("="*80)
    print("ALIGNMENT MATRIX ANALYSIS")
    print("="*80)
    print(f"Total samples: {len(ground_truth)}")
    print(f"Members: {np.sum(member_mask)} ({np.mean(member_mask)*100:.1f}%)")
    print(f"Non-members: {np.sum(non_member_mask)} ({np.mean(non_member_mask)*100:.1f}%)")
    print(f"Pseudo-experts: {n_pseudo_experts}")
    
    # 1. PER PSEUDO-EXPERT ANALYSIS
    print("\n" + "="*60)
    print("PER PSEUDO-EXPERT ANALYSIS")
    print("="*60)
    
    for k in range(n_pseudo_experts):
        print(f"\nPseudo-Expert {k+1}:")
        print("-" * 30)
        
        member_k = member_scores[:, k]
        non_member_k = non_member_scores[:, k]
        
        # Distribution statistics
        print(f"Members     - Mean: {np.mean(member_k):6.3f} ± {np.std(member_k):5.3f}")
        print(f"Non-members - Mean: {np.mean(non_member_k):6.3f} ± {np.std(non_member_k):5.3f}")
        
        # Positive/negative projections
        member_pos = np.sum(member_k > 0)
        member_neg = np.sum(member_k < 0)
        non_member_pos = np.sum(non_member_k > 0)
        non_member_neg = np.sum(non_member_k < 0)
        
        print(f"Members     - Positive: {member_pos:3d}, Negative: {member_neg:3d}")
        print(f"Non-members - Positive: {non_member_pos:3d}, Negative: {non_member_neg:3d}")
    
    # 2. OVERALL ANALYSIS
    print("\n" + "="*60)
    print("OVERALL ANALYSIS")
    print("="*60)
    
    # Average alignment score per sample
    member_avg_scores = np.mean(member_scores, axis=1)
    non_member_avg_scores = np.mean(non_member_scores, axis=1)
    
    print(f"Average alignment scores:")
    print(f"Members     - Mean: {np.mean(member_avg_scores):6.3f} ± {np.std(member_avg_scores):5.3f}")
    print(f"Non-members - Mean: {np.mean(non_member_avg_scores):6.3f} ± {np.std(non_member_avg_scores):5.3f}")
    
    return {
        'member_scores': member_scores,
        'non_member_scores': non_member_scores,
        'member_avg_scores': member_avg_scores,
        'non_member_avg_scores': non_member_avg_scores
    }

In [ ]:
def plot_heatmap_analysis(alignment_matrix, ground_truth):
    """
    Create detailed heatmap visualizations
    """
    # Sort by membership for better visualization
    sort_idx = np.argsort(ground_truth)
    sorted_matrix = alignment_matrix[sort_idx]
    sorted_labels = ground_truth[sort_idx]
    
    # Find the boundary between non-members and members
    boundary = np.sum(sorted_labels == 0)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Full heatmap
    im1 = axes[0].imshow(sorted_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].axhline(y=boundary-0.5, color='yellow', linewidth=2, label='Member/Non-member boundary')
    axes[0].set_title('Alignment Matrix (Sorted by Membership)')
    axes[0].set_xlabel('Pseudo-expert')
    axes[0].set_ylabel('Sample (0=Non-member, 1=Member)')
    plt.colorbar(im1, ax=axes[0])
    
    # Average by membership
    member_avg = np.mean(alignment_matrix[ground_truth == 1], axis=0)
    non_member_avg = np.mean(alignment_matrix[ground_truth == 0], axis=0)
    
    avg_matrix = np.array([non_member_avg, member_avg])
    im2 = axes[1].imshow(avg_matrix, aspect='auto', cmap='RdBu_r')
    axes[1].set_title('Average Alignment by Membership')
    axes[1].set_xlabel('Pseudo-expert')
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(['Non-members', 'Members'])
    plt.colorbar(im2, ax=axes[1])
    
    # Difference plot
    diff = member_avg - non_member_avg
    
    print(f"\nPseudo-expert differences (Members - Non-members):")
    for i, d in enumerate(diff):
        print(f"Expert {i+1}: {d:6.3f}")
        
    im3 = axes[2].imshow(diff.reshape(1, -1), aspect='auto', cmap='RdBu_r')
    axes[2].set_title('Difference (Members - Non-members)')
    axes[2].set_xlabel('Pseudo-expert')
    axes[2].set_yticks([0])
    axes[2].set_yticklabels(['Difference'])
    plt.colorbar(im3, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Basic analysis
results = analyze_alignment_matrix(alignment_matrix, ground_truth)

In [ ]:
# Heatmap analysis
plot_heatmap_analysis(alignment_matrix, ground_truth)

In [ ]:
# Projection distributions for different epsilon values
epsilon_values = [0.2,0.4,0.6,0.7]
for eps in epsilon_values:
    print(f"\n{'='*60}")
    print(f"ANALYSIS WITH EPSILON = {eps}")
    print(f"{'='*60}")
    plot_projection_distributions(alignment_matrix, ground_truth, epsilon=eps)

In [ ]:
plot_average_alignment(alignment_matrix,ground_truth)

In [ ]:
# No of projections should be capped at numpseudoexperts + 1 
# see the histogram of the projection values

# The gradients measure the relative loss across that axis
# It could be that datapoints trained for model earlier may not give as much gradient later on
# Can we somehow capture this?